# Global Stock Data Pipeline

Fetch and store global financial data from Yahoo Finance:
- **Stock Prices**: 1 year of daily OHLCV for major tickers (AAPL, MSFT, GOOGL…)
- **Financial Metrics**: P/E, EPS, Revenue, Market Cap, etc.
- **Macro Indicators**: VIX, Gold, Oil, Treasury Yields, FX rates

All data is stored in PostgreSQL and used by the SQL route in the RAG pipeline.

---

In [1]:
import sys
import os
from pathlib import Path
from _setup import PROJECT_ROOT

import pandas as pd
import yfinance as yf
from datetime import datetime, timezone

from storage.relational_db import RelationalDB

db = RelationalDB()
db.setup_database()
print('[OK] Connected to PostgreSQL')

[OK] Connected to PostgreSQL


## 1. Define Tickers

In [2]:
STOCK_TICKERS = [
    # US Tech
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA",
    # US Finance
    "JPM", "GS", "V", "MA", "BAC",
    # US Healthcare/Consumer
    "JNJ", "UNH", "PG", "KO",
    # Europe
    "ASML", "SAP", "SHEL", "NVS",
    # Asia
    "TSM", "BABA",
    # Energy
    "XOM", "CVX",
    # ETFs
    "SPY", "QQQ", "EEM", "VTI",
]

MACRO_TICKERS = {
    "^VIX": ("VIX (Volatility Index)", "index"),
    "^GSPC": ("S&P 500", "index"),
    "^IXIC": ("NASDAQ Composite", "index"),
    "^DJI": ("Dow Jones Industrial", "index"),
    "GC=F": ("Gold (USD/oz)", "commodity"),
    "CL=F": ("Crude Oil WTI (USD/bbl)", "commodity"),
    "BZ=F": ("Brent Oil (USD/bbl)", "commodity"),
    "EURUSD=X": ("EUR/USD", "forex"),
    "GBPUSD=X": ("GBP/USD", "forex"),
    "^TNX": ("US 10Y Treasury Yield", "rate"),
}

print(f"Stock tickers: {len(STOCK_TICKERS)}")
print(f"Macro tickers: {len(MACRO_TICKERS)}")

Stock tickers: 28
Macro tickers: 10


## 2. Fetch Stock Prices (OHLCV)

In [3]:
# Fetch a single ticker as demo
demo_ticker = "AAPL"
data = yf.download(demo_ticker, period="1mo", progress=False, auto_adjust=True)

# Flatten multi-level columns
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

print(f"\n{demo_ticker} — Last 30 days ({len(data)} rows):")
data.tail(5)


AAPL — Last 30 days (21 rows):


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-06-25,275.149994,288.799988,273.750000,287.399994,107013700
2026-06-26,283.779999,285.950012,274.209991,275.000000,261775500
2026-06-29,281.739990,288.369995,279.850006,286.730011,66427000
2026-06-30,289.359985,289.940002,280.700012,281.170013,65100200
2026-07-01,294.380005,296.589996,289.200012,293.440002,50109500


In [4]:
# ── Fetch & store prices for ALL tickers ────────────────────────────────
# NOTE: This takes ~2 minutes (Yahoo Finance rate limits)

PERIOD = "1y"  # Change to "5d" for a quick daily update
total_inserted = 0

for i, ticker in enumerate(STOCK_TICKERS, 1):
    try:
        data = yf.download(ticker, period=PERIOD, progress=False, auto_adjust=True)
        if data.empty:
            continue
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        
        records = []
        for date_idx, row in data.iterrows():
            records.append({
                "ticker": ticker,
                "date": date_idx.strftime("%Y-%m-%d"),
                "open": float(row["Open"]) if pd.notna(row["Open"]) else None,
                "high": float(row["High"]) if pd.notna(row["High"]) else None,
                "low": float(row["Low"]) if pd.notna(row["Low"]) else None,
                "close": float(row["Close"]) if pd.notna(row["Close"]) else None,
                "volume": int(row["Volume"]) if pd.notna(row["Volume"]) else None,
            })
        
        count = db.insert_prices(records)
        total_inserted += count
        print(f"  [{i:2d}/{len(STOCK_TICKERS)}] {ticker:6s} — {count} rows", end="\r")
    except Exception as e:
        print(f"  [{i:2d}/{len(STOCK_TICKERS)}] {ticker:6s} — ERROR: {e}")

print(f"\n\n[OK] Total price rows inserted: {total_inserted:,}")

  [28/28] VTI    — 246 rows

[OK] Total price rows inserted: 6,888


## 3. Fetch Financial Metrics

In [5]:
# Fetch key fundamentals for a single ticker
info = yf.Ticker("AAPL").info

key_metrics = [
    "marketCap", "trailingPE", "forwardPE", "trailingEps",
    "revenue", "netIncome", "ebitda", "profitMargins",
    "debtToEquity", "returnOnEquity", "dividendYield", "beta"
]

print(f"AAPL — Key Metrics:")
print("-" * 40)
for metric in key_metrics:
    val = info.get(metric)
    if val is not None:
        if isinstance(val, float) and val > 1e6:
            print(f"  {metric:20s}: {val:>15,.0f}")
        else:
            print(f"  {metric:20s}: {val}")

AAPL — Key Metrics:
----------------------------------------
  marketCap           : 4323663937536
  trailingPE          : 35.59613
  forwardPE           : 30.636118
  trailingEps         : 8.27
  ebitda              : 159975997440
  profitMargins       : 0.27152002
  debtToEquity        : 79.548
  returnOnEquity      : 1.4147099
  dividendYield       : 0.37
  beta                : 1.086


In [7]:
# -- Fetch metrics for all tickers and store in PostgreSQL ───────────────
# Note: if metrics already exist in the DB, this performs an UPSERT (updates values).
import time
from sqlalchemy.dialects.postgresql import insert as pg_insert
from storage.models import FinancialMetric

metrics_to_fetch = [
    "marketCap", "trailingPE", "forwardPE", "trailingEps", "forwardEps",
    "revenue", "netIncome", "ebitda", "operatingIncome", "grossProfits",
    "freeCashflow", "operatingMargins", "profitMargins",
    "debtToEquity", "returnOnEquity", "returnOnAssets",
    "beta", "dividendYield", "fiftyTwoWeekHigh", "fiftyTwoWeekLow",
]

total_metrics = 0
for i, ticker in enumerate(STOCK_TICKERS[:5], 1):  # Demo with 5 tickers
    try:
        info = yf.Ticker(ticker).info
        records = []
        for metric in metrics_to_fetch:
            value = info.get(metric)
            if value is not None:
                metric_name = ''.join(['_' + c.lower() if c.isupper() else c for c in metric]).lstrip('_')
                records.append({
                    "ticker": ticker,
                    "metric_name": metric_name,
                    "value": float(value) if isinstance(value, (int, float)) else None,
                    "period": "latest",
                })
        # Upsert: update value if the metric already exists
        with db.SessionLocal() as session:
            for rec in records:
                stmt = pg_insert(FinancialMetric).values(**rec)
                stmt = stmt.on_conflict_do_update(
                    index_elements=['ticker', 'metric_name', 'period'],
                    set_={'value': stmt.excluded.value}
                )
                session.execute(stmt)
            session.commit()
        total_metrics += len(records)
        print(f"  [{i}] {ticker:6s} -- {len(records)} metrics stored")
        time.sleep(1)  # Rate limit
    except Exception as e:
        print(f"  [{i}] {ticker:6s} -- ERROR: {e}")

print(f"\n[OK] Total metrics stored: {total_metrics}")

  [1] AAPL   -- 17 metrics stored
  [2] MSFT   -- 17 metrics stored
  [3] GOOGL  -- 17 metrics stored
  [4] AMZN   -- 16 metrics stored
  [5] NVDA   -- 17 metrics stored

[OK] Total metrics stored: 84


## 4. Fetch Macro Indicators

In [8]:
# Fetch macro data (VIX, Gold, Oil, FX, etc.)
total_macro = 0

for ticker, (name, category) in list(MACRO_TICKERS.items())[:5]:  # Demo with 5
    try:
        data = yf.download(ticker, period="1mo", progress=False, auto_adjust=True)
        if data.empty:
            continue
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        
        latest = data.iloc[-1]
        close_val = float(latest["Close"]) if pd.notna(latest["Close"]) else None
        
        if close_val:
            print(f"  {name:35s}: {close_val:>12,.2f} ({category})")
            total_macro += 1
    except Exception as e:
        print(f"  {name:35s}: ERROR — {e}")

print(f"\n[OK] Fetched {total_macro} macro indicators")

  VIX (Volatility Index)             :        16.70 (index)
  S&P 500                            :     7,483.23 (index)
  NASDAQ Composite                   :    26,040.03 (index)
  Dow Jones Industrial               :    52,305.24 (index)
  Gold (USD/oz)                      :     4,076.30 (commodity)

[OK] Fetched 5 macro indicators


## 5. Query the Database

In [9]:
# Check what's in the DB
prices_count = db.execute_sql("SELECT COUNT(*) as cnt FROM stock_prices")[0]["cnt"]
metrics_count = db.execute_sql("SELECT COUNT(*) as cnt FROM financial_metrics")[0]["cnt"]

print(f"Database Contents:")
print(f"  stock_prices:      {prices_count:,} rows")
print(f"  financial_metrics: {metrics_count:,} rows")

# Sample prices
print("\nLatest prices (sample):")
sample = db.execute_sql(
    "SELECT ticker, date, close, volume FROM stock_prices "
    "ORDER BY date DESC LIMIT 10"
)
pd.DataFrame(sample)

Database Contents:
  stock_prices:      7,253 rows
  financial_metrics: 470 rows

Latest prices (sample):


,ticker,date,close,volume
0,BVC.ADH,2026-07-02,34.25,83656
1,BVC.ADI,2026-07-02,398.00,42271
2,BVC.ATL,2026-07-02,126.20,50404
3,BVC.ATW,2026-07-02,690.00,1530159
4,BVC.BCP,2026-07-02,266.00,1295446
5,BVC.CIH,2026-07-02,346.00,54585
6,BVC.CSR,2026-07-02,194.35,107276
7,BVC.DHO,2026-07-02,60.00,6615
8,BVC.IAM,2026-07-02,92.20,722051
9,BVC.LES,2026-07-02,350.15,350
